# Graph Path Algorithms Solutions

In [ ]:
import neo4j

import pandas as pd

from IPython.display import display

In [ ]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [ ]:
session = driver.session(database="neo4j")

In [ ]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [ ]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [ ]:
def my_neo4j_nodes_relationships():
    "print all the nodes and relationships"
   
    print("-------------------------")
    print("  Nodes:")
    print("-------------------------")
    
    query = """
        match (n) 
        return n.name as node_name, labels(n) as labels
        order by n.name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    display(df)
    
    print("-------------------------")
    print("  Relationships:")
    print("-------------------------")
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.name as node_name_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.name as node_name_2, labels(n2) as node_2_labels
        order by node_name_1, node_name_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    display(df)
    
    density = (2 * number_relationships) / (number_nodes * (number_nodes - 1))
    
    print("-------------------------")
    print("  Density:", f'{density:.1f}')
    print("-------------------------")
    

In [ ]:
my_neo4j_wipe_out_database()

query = """

CREATE
  (seattle:Station {name: 'Seattle', latitude: 47.6062, longitude: -122.3321}),
  (berkeley:Station {name: 'Berkeley', latitude: 37.8715, longitude: -122.2730}),
  (losangeles:Station {name: 'Los Angeles', latitude: 34.0522, longitude: -118.2437}),
  (denver:Station {name: 'Denver', latitude: 39.7392, longitude: -104.9903}),
  (dallas:Station {name: 'Dallas', latitude: 32.7767, longitude: -96.7970}),
  (chicago:Station {name: 'Chicago', latitude: 41.8781, longitude: -87.6298}),
  (newyork:Station {name: 'New York', latitude: 40.7128, longitude: -74.0060}),
  (washington:Station {name: 'Washington', latitude: 38.9072, longitude: -77.0369}),
  (miami:Station {name: 'Miami', latitude: 25.7617, longitude: -80.1918}),
  (seattle)-[:TRACK {track_miles: 798}]->(berkeley),
  (berkeley)-[:TRACK {track_miles: 798}]->(seattle),
  (seattle)-[:TRACK {track_miles: 1303}]->(denver),
  (denver)-[:TRACK {track_miles: 1303}]->(seattle),
  (berkeley)-[:TRACK {track_miles: 1240}]->(denver),
  (denver)-[:TRACK {track_miles: 1240}]->(berkeley),
  (berkeley)-[:TRACK {track_miles: 376}]->(losangeles),
  (losangeles)-[:TRACK {track_miles: 376}]->(berkeley),
  (losangeles)-[:TRACK {track_miles: 1436}]->(dallas),
  (dallas)-[:TRACK {track_miles: 1436}]->(losangeles),
  (denver)-[:TRACK {track_miles: 1003}]->(chicago),
  (chicago)-[:TRACK {track_miles: 1003}]->(denver),
  (denver)-[:TRACK {track_miles: 794}]->(dallas),
  (dallas)-[:TRACK {track_miles: 794}]->(denver),
  (chicago)-[:TRACK {track_miles: 794}]->(newyork),
  (newyork)-[:TRACK {track_miles: 794}]->(chicago),
  (dallas)-[:TRACK {track_miles: 1329}]->(washington),
  (washington)-[:TRACK {track_miles: 1329}]->(dallas),
  (newyork)-[:TRACK {track_miles: 226}]->(washington),
  (washington)-[:TRACK {track_miles: 226}]->(newyork),
  (washington)-[:TRACK {track_miles: 1053}]->(miami),
  (miami)-[:TRACK {track_miles: 1053}]->(washington)
  
"""

session.run(query)

In [ ]:
my_neo4j_nodes_relationships()

## You try it - find the shortest path using Dijkstra for Los Angeles to New York and return New York to Los Angeles; solutions in graph_path_algorithm_solutions

In [ ]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'TRACK', {relationshipProperties: 'track_miles'})"
session.run(query)

In [ ]:
query = """

MATCH (source:Station {name: $source}), (target:Station {name: $target})
CALL gds.shortestPath.dijkstra.stream(
    'ds_graph', 
    { sourceNode: source, 
      targetNode: target, 
      relationshipWeightProperty: 'track_miles'
    }
)
YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path
RETURN
    gds.util.asNode(sourceNode).name AS from,
    gds.util.asNode(targetNode).name AS to,
    totalCost,
    [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodes,
    costs
ORDER BY index

"""

source = "Los Angeles"
target = "New York"

my_neo4j_run_query_pandas(query, source=source, target=target)

In [ ]:
query = """

MATCH (source:Station {name: $source}), (target:Station {name: $target})
CALL gds.shortestPath.dijkstra.stream(
    'ds_graph', 
    { sourceNode: source, 
      targetNode: target, 
      relationshipWeightProperty: 'track_miles'
    }
)
YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path
RETURN
    gds.util.asNode(sourceNode).name AS from,
    gds.util.asNode(targetNode).name AS to,
    totalCost,
    [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodes,
    costs
ORDER BY index

"""

source = "New York"
target = "Los Angeles"

my_neo4j_run_query_pandas(query, source=source, target=target)

## You try it - find the shortest path using A Star for Los Angeles to New York and return New York to Los Angeles; solutions in graph_path_algorithm_solutions

In [ ]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

query = """
CALL gds.graph.project('ds_graph', 'Station', 'TRACK', 
                      {nodeProperties: ['latitude', 'longitude'],
                      relationshipProperties: 'track_miles'})
"""
session.run(query)

In [ ]:
query = """

MATCH (source:Station {name: $source}), (target:Station {name: $target})
CALL gds.shortestPath.astar.stream(
    'ds_graph', 
    { sourceNode: source,
      targetNode: target,
      latitudeProperty: 'latitude',
      longitudeProperty: 'longitude',
      relationshipWeightProperty: 'track_miles'
     }
)
YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path
RETURN
    index,
    gds.util.asNode(sourceNode).name AS sourceNodeName,
    gds.util.asNode(targetNode).name AS targetNodeName,
    totalCost,
    [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodeNames,
    costs
ORDER BY index

"""

source = "Los Angeles"
target = "New York"

my_neo4j_run_query_pandas(query, source=source, target=target)


In [ ]:
query = """

MATCH (source:Station {name: $source}), (target:Station {name: $target})
CALL gds.shortestPath.astar.stream(
    'ds_graph', 
    { sourceNode: source,
      targetNode: target,
      latitudeProperty: 'latitude',
      longitudeProperty: 'longitude',
      relationshipWeightProperty: 'track_miles'
    }
)
YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path
RETURN
    index,
    gds.util.asNode(sourceNode).name AS sourceNodeName,
    gds.util.asNode(targetNode).name AS targetNodeName,
    totalCost,
    [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodeNames,
    costs
ORDER BY index

"""

source = "New York"
target = "Los Angeles"

my_neo4j_run_query_pandas(query, source=source, target=target)

## You try it - find the shortest path using  Yen's (for up to 10 paths) for Los Angeles to New York and return New York to Los Angeles; solutions in graph_path_algorithm_solutions

In [ ]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'TRACK', {relationshipProperties: 'track_miles'})"
session.run(query)

In [ ]:
query = """

MATCH (source:Station {name: $source}), (target:Station {name: $target})
CALL gds.shortestPath.yens.stream(
    'ds_graph', 
    { sourceNode: source,
      targetNode: target,
      k: $k,
      relationshipWeightProperty: 'track_miles'
    }
)
YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path
RETURN
    index,
    gds.util.asNode(sourceNode).name AS sourceNodeName,
    gds.util.asNode(targetNode).name AS targetNodeName,
    totalCost,
    [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodeNames,
    costs
ORDER BY index

"""

source = "Los Angeles"
target = "New York"
k = 10

my_neo4j_run_query_pandas(query, source=source, target=target, k=k)


In [ ]:
query = """

MATCH (source:Station {name: $source}), (target:Station {name: $target})
CALL gds.shortestPath.yens.stream(
    'ds_graph', 
    { sourceNode: source,
      targetNode: target,
      k: $k,
      relationshipWeightProperty: 'track_miles'
    }
)
YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path
RETURN
    index,
    gds.util.asNode(sourceNode).name AS sourceNodeName,
    gds.util.asNode(targetNode).name AS targetNodeName,
    totalCost,
    [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodeNames,
    costs
ORDER BY index

"""

source = "New York"
target = "Los Angeles"
k = 10

my_neo4j_run_query_pandas(query, source=source, target=target, k=k)


## You try it - find the single source shortest paths from Los Angeles; solutions in graph_path_algorithms_solutions

In [ ]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'TRACK', {relationshipProperties: 'track_miles'})"
session.run(query)

In [ ]:
query = """

MATCH (n:Station {name: $source})
CALL gds.allShortestPaths.delta.stream('ds_graph', 
                                       {sourceNode: n,
                                        relationshipWeightProperty: 'track_miles',
                                        delta: 3.0
                                       }
                                      )
                                        
YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs
RETURN
    index,
    gds.util.asNode(sourceNode).name AS sourceNodeName,
    gds.util.asNode(targetNode).name AS targetNodeName,
    totalCost,
    [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodeNames,
    costs
ORDER BY index

"""

source = "Los Angeles"

my_neo4j_run_query_pandas(query, source=source)

In [ ]:
def my_get_node_list():
    "get a list of nodes in the current graph"
    
    query = "match (n) return n.name as name"
    
    result = session.run(query)
    
    node_list = []
    
    for r in result:
        node_list.append(r["name"])
        
    node_list = sorted(node_list)
    
    return node_list

In [ ]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'TRACK', {relationshipProperties: 'track_miles'})"
session.run(query)

In [ ]:
query = """

MATCH (source:Station {name: $source}), (target:Station {name: $target})
CALL gds.shortestPath.dijkstra.stream(
    'ds_graph', 
    { sourceNode: source, 
      targetNode: target, 
      relationshipWeightProperty: 'track_miles'
    }
)
YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path
RETURN
    gds.util.asNode(sourceNode).name AS from,
    gds.util.asNode(targetNode).name AS to,
    totalCost,
    [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodes,
    costs
ORDER BY index

"""

nodes = my_get_node_list()

single_source_shortest_paths = []

source = "Los Angeles"
    
for target in nodes:

    if source != target:

        result = session.run(query, source=source, target=target)

        for r in result:

            single_source_shortest_paths.append([r["from"], r["to"], r["totalCost"], r["nodes"], r["costs"]])
                
for path in single_source_shortest_paths:
    
    print(path)
                      

## You try it - find the Minimum Spanning Tree with the source at Los Angeles

In [ ]:
def my_neo4j_wipe_out_mst_relationships():
    "wipe out mst relationships"
    
    query = "match (node)-[relationship:MST]->() delete relationship"
    session.run(query)

In [ ]:
my_neo4j_wipe_out_mst_relationships()

In [ ]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

query = """

CALL gds.graph.project('ds_graph', 'Station', 
                        {
                            TRACK: {
                                properties: 'track_miles',
                                orientation: 'UNDIRECTED'
                            }
                        }
                       )
                                
"""

session.run(query)

In [ ]:
query = """

MATCH (n:Station {name: $source})
CALL gds.spanningTree.write('ds_graph',
                                          {sourceNode: n,
                                           relationshipWeightProperty: 'track_miles',
                                           writeProperty: 'writeCost',
                                           writeRelationshipType: 'MST'
                                          }
                                         )
YIELD preProcessingMillis, computeMillis, writeMillis, effectiveNodeCount
RETURN preProcessingMillis, computeMillis, writeMillis, effectiveNodeCount;

"""

source = "Los Angeles"

my_neo4j_run_query_pandas(query, source=source)

In [ ]:
query = """

MATCH path = (n:Station {name: $source})-[:MST*]-()
WITH relationships(path) AS rels
UNWIND rels AS rel
WITH DISTINCT rel AS rel
RETURN startNode(rel).name AS source, endNode(rel).name AS destination, rel.writeCost AS cost

"""

source = "Los Angeles"

my_neo4j_run_query_pandas(query, source=source)

## You try it - take a random walk from the Los Angeles node with path size of 10; solution is in graph_path_algorithms_solutions

In [ ]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'TRACK', {relationshipProperties: 'track_miles'})"
session.run(query)

In [ ]:
query = """

MATCH (home:Station {name: $source})
WITH COLLECT(home) as sourceNodes
CALL gds.randomWalk.stream('ds_graph',
                                {sourceNodes: sourceNodes,
                                 walkLength: $path_size,
                                 walksPerNode: 1,
                                 concurrency: 1
                                }
                               )
YIELD nodeIds
UNWIND nodeIds AS nodeId
RETURN gds.util.asNode(nodeId).name AS node

"""

source = "Los Angeles"

path_size = 10

my_neo4j_run_query_pandas(query, source=source, path_size=path_size)